In [1]:
import wandb
import pandas as pd
import numpy as np

import json

from helpers import *

api = wandb.Api()

## IAUC DAUC

In [2]:
iauc_dauc_runs = api.runs(
    "pasqualedem/affex",
    filters={
        "$and": [
            {"config.metric.n_steps": 75},
            {"tags": {"$nin": ["Reverse"]}},
            {"$or": [
                {'config.dataset.datasets.val_pascal5i_N1K5.name': "pascal"},
                {'config.dataset.datasets.val_coco20i_N1K5.name': "coco"},
            ]}
        ]
    }
)

In [3]:
iauc_dauc_runs_df = get_runs_df(iauc_dauc_runs)
len(iauc_dauc_runs_df)

69

In [4]:
res_df = iauc_dauc_runs_df.copy()

dataset_names = ["val_pascal5i_N1K5", "val_coco20i_N1K5"]

dataset_cols = [f'dataset.datasets.{name}.name' for name in dataset_names]
res_df["Dataset"] = res_df[dataset_cols].fillna("").sum(axis=1)

iauc_cols = [f"{name}_iauc" for name in dataset_names]
dauc_cols = [f"{name}_dauc" for name in dataset_names]
assert res_df[iauc_cols].isna().any(axis=1).all(), "More than one dataset is present in the same run, please check the dataset columns."
assert res_df[dauc_cols].isna().any(axis=1).all(), "More than one dataset is present in the same run, please check the dataset columns."


res_df = res_df.rename(columns=renamings_dict)

res_df["Explanation Method"] = res_df["Explanation Method"].replace(value_renamings_dict)
res_df["Dataset"] = res_df["Dataset"].replace(value_renamings_dict)
res_df["Model"] = res_df["Model"].replace(value_renamings_dict)


res_df["IAUC"] = res_df[iauc_cols].fillna(0).sum(axis=1)
res_df["DAUC"] = res_df[dauc_cols].fillna(0).sum(axis=1)

res_df["Diff."] = res_df["IAUC"] - res_df["DAUC"]

cols = ["Model", "Explanation Method", "Dataset"]
res_df = res_df[cols + ["IAUC", "DAUC", "Diff."]]


res_df = res_df.pivot_table(
    index=['Model', 'Explanation Method'],
    columns='Dataset',
    values=['IAUC', 'DAUC', 'Diff.']
).swaplevel(axis=1).sort_index(axis=1)


new_cols  = sorted(res_df.columns, key=lambda x: (dataset_order[x[0]], metric_order[x[1]]))
res_df = res_df[new_cols]
# Sort by Model and then by Explanation Method using the provided orderings
res_df = res_df.sort_values(
    by=['Model', 'Explanation Method'],  # this ensures level names exist and are used
    key=lambda x: x.map(model_order) if x.name == 'Model' else x.map(method_order)
)

iauc_dauc_df = (res_df*100).round(2)

iauc_dauc_df

Dataset                      COCO 20^i               Pascal 5^i              
                                  IAUC   DAUC  Diff.       IAUC   DAUC  Diff.
Model  Explanation Method                                                    
DCAMA  Random                    60.74  60.50   0.23      77.25  77.06   0.19
       Gaussian Noise Mask       52.32  15.46  36.86      74.67  23.36  51.31
       Saliency                  64.70  53.36  11.34      78.44  72.51   5.93
       Integrated Gradients      64.25  57.23   7.02      79.16  75.18   3.99
       Guided IG                 50.77  45.10   5.67      71.74  70.35   1.39
       Blur IG                   52.40  41.63  10.78      73.82  65.80   8.01
       XRAI                      54.45  38.72  15.73      75.52  60.98  14.55
       Deep Lift                 58.78  57.24   1.54      76.84  75.69   1.15
       LIME                      53.18  48.44   4.73      74.77  72.20   2.57
       Unmasked AffEx (ours)     67.71  41.55  26.16      83.50  56.72  26.78
       Masked AffEx (ours)       52.32  15.06  37.26      73.16  21.19  51.97
       Signed AffEx (ours)       53.24  23.77  29.48      75.46  45.45  30.01
DMTNet Random                    33.65  33.51   0.14      37.90  37.66   0.25
       Gaussian Noise Mask       39.08  15.71  23.37      54.53  24.41  30.12
       Saliency                  53.10  43.22   9.88      61.85  51.66  10.19
       Integrated Gradients      39.18  32.14   7.04      50.98  43.84   7.14
       Blur IG                   35.04  20.32  14.72      47.83  33.82  14.01
       XRAI                      38.19  26.55  11.64      55.82  45.72  10.09
       LIME                      36.83  32.44   4.39      54.76  49.80   4.96
       Unmasked AffEx (ours)     59.79  49.26  10.53      66.59  47.57  19.02
       Masked AffEx (ours)       40.52  16.38  24.14      58.47  26.87  31.60
       Signed AffEx (ours)       38.85  18.81  20.03      57.58  35.68  21.90

In [5]:
from tabulate import tabulate

latex_tab = tabulate(res_df, headers='keys', tablefmt='latex_booktabs', showindex=True)
print(latex_tab)

\begin{tabular}{lrrrrrr}
\toprule
                                     &   ('COCO 20\^{}i', 'IAUC') &   ('COCO 20\^{}i', 'DAUC') &   ('COCO 20\^{}i', 'Diff.') &   ('Pascal 5\^{}i', 'IAUC') &   ('Pascal 5\^{}i', 'DAUC') &   ('Pascal 5\^{}i', 'Diff.') \\
\midrule
 ('DCAMA', 'Random')                 &                0.607366 &                0.60505  &               0.00231661 &                 0.772463 &                 0.770567 &                0.00189589 \\
 ('DCAMA', 'Gaussian Noise Mask')    &                0.523201 &                0.154589 &               0.368612   &                 0.746695 &                 0.233592 &                0.513103   \\
 ('DCAMA', 'Saliency')               &                0.646997 &                0.533582 &               0.113415   &                 0.784401 &                 0.725112 &                0.0592886  \\
 ('DCAMA', 'Integrated Gradients')   &                0.642501 &                0.572299 &               0.0702024  &                 0

## IAUC DAUC MIOU

In [6]:
iauc_dauc_runs = api.runs(
    "pasqualedem/affex",
    filters={
        "$and": [
            {"config.metric.n_steps": 75},
            {"tags": {"$nin": ["Reverse"]}},
            {"$or": [
                {'config.dataset.datasets.val_pascal5i_N1K5.name': "pascal"},
                {'config.dataset.datasets.val_coco20i_N1K5.name': "coco"},
            ]},
            {"config.metric.measure": "miou"}
        ]
    }
)

In [7]:
iauc_dauc_runs_df = get_runs_df(iauc_dauc_runs)
len(iauc_dauc_runs_df)

47

In [8]:
res_df = iauc_dauc_runs_df.copy()

dataset_names = [
    "val_pascal5i_N1K5",
    "val_coco20i_N1K5"
    ]

dataset_cols = [f'dataset.datasets.{name}.name' for name in dataset_names]
res_df["Dataset"] = res_df[dataset_cols].fillna("").sum(axis=1)

iauc_cols = [f"{name}_iauc" for name in dataset_names]
dauc_cols = [f"{name}_dauc" for name in dataset_names]
assert res_df[iauc_cols].isna().any(axis=1).all(), "More than one dataset is present in the same run, please check the dataset columns."
assert res_df[dauc_cols].isna().any(axis=1).all(), "More than one dataset is present in the same run, please check the dataset columns."

res_df = res_df.rename(columns=renamings_dict)

res_df["Explanation Method"] = res_df["Explanation Method"].replace(value_renamings_dict)
res_df["Dataset"] = res_df["Dataset"].replace(value_renamings_dict)
res_df["Model"] = res_df["Model"].replace(value_renamings_dict)


res_df["IAUC"] = res_df[iauc_cols].fillna(0).sum(axis=1)
res_df["DAUC"] = res_df[dauc_cols].fillna(0).sum(axis=1)

res_df["Diff."] = res_df["IAUC"] - res_df["DAUC"]

cols = ["Model", "Explanation Method", "Dataset"]
res_df = res_df[cols + ["IAUC", "DAUC", "Diff."]]

res_df = res_df.pivot_table(
    index=['Model', 'Explanation Method'],
    columns='Dataset',
    values=['IAUC', 'DAUC', 'Diff.']
).swaplevel(axis=1).sort_index(axis=1)


new_cols  = sorted(res_df.columns, key=lambda x: (dataset_order[x[0]], metric_order[x[1]]))
res_df = res_df[new_cols]
# Sort by Model and then by Explanation Method using the provided orderings
res_df = res_df.sort_values(
    by=['Model', 'Explanation Method'],  # this ensures level names exist and are used
    key=lambda x: x.map(model_order) if x.name == 'Model' else x.map(method_order)
)

iauc_dauc_df = (res_df*100).round(2)

iauc_dauc_df

Dataset                      COCO 20^i               Pascal 5^i              
                                  IAUC   DAUC  Diff.       IAUC   DAUC  Diff.
Model  Explanation Method                                                    
DCAMA  Random                    47.50  46.85   0.65      71.35  70.77   0.59
       Gaussian Noise Mask       52.32  15.46  36.86      74.67  23.36  51.31
       Saliency                  52.02  41.10  10.92      72.99  66.73   6.27
       Integrated Gradients      49.23  45.97   3.27      72.38  70.22   2.16
       Guided IG                 50.77  45.10   5.67      71.74  70.35   1.39
       Blur IG                   52.40  41.63  10.78      73.82  65.80   8.01
       XRAI                      54.45  38.72  15.73      75.52  60.98  14.55
       Deep Lift                 49.78  48.51   1.27      72.72  71.66   1.06
       LIME                      53.18  48.44   4.73      74.77  72.20   2.57
       Unmasked AffEx (ours)     54.85  28.26  26.59      76.23  46.34  29.89
       Masked AffEx (ours)       52.32  15.06  37.26      73.16  21.19  51.97
       Signed AffEx (ours)       53.24  23.77  29.48      75.46  45.45  30.01
DMTNet Random                    21.21  20.98   0.23      29.01  28.46   0.55
       Gaussian Noise Mask       39.08  15.71  23.37      54.53  24.41  30.12
       Saliency                  36.51  26.79   9.72      52.43  41.22  11.21
       Integrated Gradients      26.25  19.47   6.78      37.13  29.08   8.05
       Blur IG                   35.04  20.32  14.72      47.83  33.82  14.01
       XRAI                      38.19  26.55  11.64      55.82  45.72  10.09
       LIME                      36.83  32.44   4.39      54.76  49.80   4.96
       Unmasked AffEx (ours)     40.10  17.09  23.01      58.16  28.81  29.35
       Masked AffEx (ours)       40.52  16.38  24.14      58.47  26.87  31.60
       Signed AffEx (ours)       38.85  18.81  20.03      57.58  35.68  21.90

In [9]:
from tabulate import tabulate

latex_tab = tabulate(res_df, headers='keys', tablefmt='latex_booktabs', showindex=True)
print(latex_tab)

\begin{tabular}{lrrrrrr}
\toprule
                                     &   ('COCO 20\^{}i', 'IAUC') &   ('COCO 20\^{}i', 'DAUC') &   ('COCO 20\^{}i', 'Diff.') &   ('Pascal 5\^{}i', 'IAUC') &   ('Pascal 5\^{}i', 'DAUC') &   ('Pascal 5\^{}i', 'Diff.') \\
\midrule
 ('DCAMA', 'Random')                 &                0.474994 &                0.468504 &               0.00648971 &                 0.713538 &                 0.707666 &                0.00587256 \\
 ('DCAMA', 'Gaussian Noise Mask')    &                0.523201 &                0.154589 &               0.368612   &                 0.746695 &                 0.233592 &                0.513103   \\
 ('DCAMA', 'Saliency')               &                0.52022  &                0.410973 &               0.109247   &                 0.729932 &                 0.667256 &                0.0626761  \\
 ('DCAMA', 'Integrated Gradients')   &                0.492338 &                0.459666 &               0.0326719  &                 0

## IAUC %

In [10]:
iauc_p_runs = api.runs(
    "pasqualedem/affex",
    filters={
        "$and": [
            {"config.metric.percentage": {"$ne": None}},
            {"tags": {"$nin": ["Reverse"]}},
            {"$or": [
                {'config.dataset.datasets.val_pascal5i_N1K5.name': "pascal"},
                {'config.dataset.datasets.val_coco20i_N1K5.name': "coco"},
            ]},
            {"config.metric.explanation_size": None},
        ]
    }
)

In [11]:
iauc_p_runs_df = get_runs_df(iauc_p_runs)
len(iauc_p_runs_df)

190

In [12]:
res_df = iauc_p_runs_df.copy()

# Remove ablation studies
res_df = res_df[res_df["explainer.explanation_size"].isna()]
res_df = res_df[(res_df["explainer.aggregation_method"].isna())]
res_df = res_df[
    (res_df["explainer.method"].isna())
    & (res_df["explainer.image_weight"].isna())
]
res_df = res_df[
    (res_df["explainer.model_type"].isna())
    & (res_df["explainer.kernel_width"].isna())
    & (res_df["explainer.slic_num_segments"].isna())
]
res_df = res_df[
    (res_df["explainer.use_softmax"].isna())
]

dataset_names = ["val_pascal5i_N1K5", "val_coco20i_N1K5"]

dataset_cols = [f"dataset.datasets.{name}.name" for name in dataset_names]
res_df["Dataset"] = res_df[dataset_cols].fillna("").sum(axis=1)

iauc_cols = [f"{name}_iauc" for name in dataset_names]
assert (
    res_df[iauc_cols].isna().any(axis=1).all()
), "More than one dataset is present in the same run, please check the dataset columns."


renamings_dict = {
    "explainer.name": "Explanation Method",
    "config.metric.n_steps": "N. Steps",
    "config.metric.iauc": "IAUC",
    "config.metric.dauc": "DAUC",
    "model": "Model",
    "metric.percentage": "Percentage",
    "metric.loss": "Loss",
}
res_df = res_df.rename(columns=renamings_dict)

res_df["Explanation Method"] = res_df["Explanation Method"].replace(
    value_renamings_dict
)
res_df["Dataset"] = res_df["Dataset"].replace(value_renamings_dict)
res_df["Model"] = res_df["Model"].replace(value_renamings_dict)


res_df["iAUC"] = res_df[iauc_cols].fillna(0).sum(axis=1)
res_df["Loss"] = res_df["Loss"].fillna(False)

cols = ["Model", "Explanation Method", "Dataset", "Percentage", "Loss"]
res_df = res_df[cols + ["iAUC"]]

# Assert that for each (Model, Explanation Method, Dataset, Percentage, Loss) there is only one run
assert (
    res_df.duplicated(
        subset=["Model", "Explanation Method", "Dataset", "Percentage", "Loss"]
    ).sum()
    == 0
), "There are duplicated runs for the same (Model, Explanation Method, Dataset, Percentage, Loss). Please check the data."

print(res_df.to_string())
res_df = res_df.pivot_table(
    index=["Model", "Explanation Method"],
    columns=["Dataset", "Percentage", "Loss"],
    values="iAUC",
).sort_index(axis=1)
perc_map = {False: "IAUC@", True: "mIoULoss@"}

# Flatten the MultiIndex columns
# Collapse last two levels into one
res_df.columns = pd.MultiIndex.from_tuples(
    [(lvl0, f"{perc_map[lvl2]}{lvl1}") for lvl0, lvl1, lvl2 in res_df.columns]
)


# Sort by Model and then by Explanation Method using the provided orderings
res_df = res_df.sort_values(
    by=["Model", "Explanation Method"],  # this ensures level names exist and are used
    key=lambda x: x.map(model_order) if x.name == "Model" else x.map(method_order),
)

iauc_p_df = (res_df * 100).round(2)
iauc_p_df.fillna("N/A", inplace=True)

iauc_p_df

      Model     Explanation Method     Dataset  Percentage   Loss      iAUC
0    DMTNet                 Random  Pascal 5^i        0.01  False  0.603507
1     DCAMA  Unmasked AffEx (ours)  Pascal 5^i        0.01  False  0.668181
2    DMTNet  Unmasked AffEx (ours)  Pascal 5^i        0.01  False  0.667953
3     DCAMA                 Random  Pascal 5^i        0.01  False  0.417120
4     DCAMA  Unmasked AffEx (ours)  Pascal 5^i        0.05   True  0.057156
5    DMTNet                 Random  Pascal 5^i        0.05   True  0.460067
6     DCAMA                 Random  Pascal 5^i        0.05   True  0.334168
7     DCAMA  Unmasked AffEx (ours)  Pascal 5^i        0.01   True  0.137830
8     DCAMA                 Random  Pascal 5^i        0.01   True  0.552434
9    DMTNet  Unmasked AffEx (ours)  Pascal 5^i        0.05   True  0.053162
10   DMTNet                 Random  Pascal 5^i        0.01   True  0.338943
11   DMTNet  Unmasked AffEx (ours)  Pascal 5^i        0.01   True  0.133373
12    DCAMA 

/tmp/ipykernel_949270/440082267.py:49: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  res_df["Loss"] = res_df["Loss"].fillna(False)


COCO 20^i                             Pascal 5^i  \
                             IAUC@0.01 mIoULoss@0.01 mIoULoss@0.05  IAUC@0.01   
Model  Explanation Method                                                       
DCAMA  Random                    43.36         41.42         31.62      41.71   
       Gaussian Noise Mask       53.80         31.08         17.71      55.32   
       Saliency                  44.97         37.01         23.73      44.32   
       Integrated Gradients      45.63         34.18         22.25      42.94   
       Guided IG                 51.25         30.17         21.65      49.08   
       Blur IG                   44.25         36.69         22.84      42.23   
       XRAI                      51.98         26.59         12.08      56.12   
       Deep Lift                 44.80         36.67         24.07      44.19   
       LIME                      47.98         32.44         15.29      49.19   
       Unmasked AffEx (ours)     60.82         17.41          8.85      66.82   
       Masked AffEx (ours)       62.60         16.43         12.27      67.91   
       Signed AffEx (ours)       60.49         19.40         13.67      66.60   
DMTNet Random                    60.36         24.93         31.88      60.35   
       Gaussian Noise Mask       59.35         20.69         13.26      59.90   
       Saliency                  63.09         21.05         15.77      64.85   
       Integrated Gradients      62.51         21.63         20.32      63.99   
       Blur IG                   62.17         21.54         20.20      63.62   
       XRAI                      68.42         16.48         10.73      69.78   
       LIME                      70.72         19.20         11.36      70.56   
       Unmasked AffEx (ours)     67.30         10.53          4.49      66.80   
       Masked AffEx (ours)       67.23          9.95          3.10      66.94   
       Signed AffEx (ours)       67.16         10.49          3.72      68.05   

                                                          
                             mIoULoss@0.01 mIoULoss@0.05  
Model  Explanation Method                                 
DCAMA  Random                        55.24         33.42  
       Gaussian Noise Mask           35.65         15.86  
       Saliency                      46.42         24.46  
       Integrated Gradients          45.01         23.26  
       Guided IG                     42.53         23.19  
       Blur IG                       46.86         21.53  
       XRAI                          26.25          8.55  
       Deep Lift                     45.82         22.83  
       LIME                          36.71         13.40  
       Unmasked AffEx (ours)         13.78          5.72  
       Masked AffEx (ours)           13.71          7.09  
       Signed AffEx (ours)           13.93          7.82  
DMTNet Random                        33.89         46.01  
       Gaussian Noise Mask           33.24         31.00  
       Saliency                      29.85         25.49  
       Integrated Gradients          30.31         33.07  
       Blur IG                       29.77         33.13  
       XRAI                          20.03         12.88  
       LIME                          23.46         12.43  
       Unmasked AffEx (ours)         13.34          5.32  
       Masked AffEx (ours)           12.78          4.62  
       Signed AffEx (ours)           13.16          6.38

---

In [13]:
# joining iauc_dauc_df and iauc_p_df on index
final = iauc_dauc_df.join(iauc_p_df)

metric_order = {
    'IAUC': 0,
    'DAUC': 1,
    'Diff.': 2,
    'IAUC@0.01': 3,
    'IAUC@0.05': 4,
    'mIoULoss@0.01': 5,
    'mIoULoss@0.05': 6,
}

new_cols  = sorted(final.columns, key=lambda x: (dataset_order[x[0]], metric_order[x[1]]))
final = final[new_cols]
final

COCO 20^i                                        \
                                  IAUC   DAUC  Diff. IAUC@0.01 mIoULoss@0.01   
Model  Explanation Method                                                      
DCAMA  Random                    47.50  46.85   0.65     43.36         41.42   
       Gaussian Noise Mask       52.32  15.46  36.86     53.80         31.08   
       Saliency                  52.02  41.10  10.92     44.97         37.01   
       Integrated Gradients      49.23  45.97   3.27     45.63         34.18   
       Guided IG                 50.77  45.10   5.67     51.25         30.17   
       Blur IG                   52.40  41.63  10.78     44.25         36.69   
       XRAI                      54.45  38.72  15.73     51.98         26.59   
       Deep Lift                 49.78  48.51   1.27     44.80         36.67   
       LIME                      53.18  48.44   4.73     47.98         32.44   
       Unmasked AffEx (ours)     54.85  28.26  26.59     60.82         17.41   
       Masked AffEx (ours)       52.32  15.06  37.26     62.60         16.43   
       Signed AffEx (ours)       53.24  23.77  29.48     60.49         19.40   
DMTNet Random                    21.21  20.98   0.23     60.36         24.93   
       Gaussian Noise Mask       39.08  15.71  23.37     59.35         20.69   
       Saliency                  36.51  26.79   9.72     63.09         21.05   
       Integrated Gradients      26.25  19.47   6.78     62.51         21.63   
       Blur IG                   35.04  20.32  14.72     62.17         21.54   
       XRAI                      38.19  26.55  11.64     68.42         16.48   
       LIME                      36.83  32.44   4.39     70.72         19.20   
       Unmasked AffEx (ours)     40.10  17.09  23.01     67.30         10.53   
       Masked AffEx (ours)       40.52  16.38  24.14     67.23          9.95   
       Signed AffEx (ours)       38.85  18.81  20.03     67.16         10.49   

                                           Pascal 5^i                          \
                             mIoULoss@0.05       IAUC   DAUC  Diff. IAUC@0.01   
Model  Explanation Method                                                       
DCAMA  Random                        31.62      71.35  70.77   0.59     41.71   
       Gaussian Noise Mask           17.71      74.67  23.36  51.31     55.32   
       Saliency                      23.73      72.99  66.73   6.27     44.32   
       Integrated Gradients          22.25      72.38  70.22   2.16     42.94   
       Guided IG                     21.65      71.74  70.35   1.39     49.08   
       Blur IG                       22.84      73.82  65.80   8.01     42.23   
       XRAI                          12.08      75.52  60.98  14.55     56.12   
       Deep Lift                     24.07      72.72  71.66   1.06     44.19   
       LIME                          15.29      74.77  72.20   2.57     49.19   
       Unmasked AffEx (ours)          8.85      76.23  46.34  29.89     66.82   
       Masked AffEx (ours)           12.27      73.16  21.19  51.97     67.91   
       Signed AffEx (ours)           13.67      75.46  45.45  30.01     66.60   
DMTNet Random                        31.88      29.01  28.46   0.55     60.35   
       Gaussian Noise Mask           13.26      54.53  24.41  30.12     59.90   
       Saliency                      15.77      52.43  41.22  11.21     64.85   
       Integrated Gradients          20.32      37.13  29.08   8.05     63.99   
       Blur IG                       20.20      47.83  33.82  14.01     63.62   
       XRAI                          10.73      55.82  45.72  10.09     69.78   
       LIME                          11.36      54.76  49.80   4.96     70.56   
       Unmasked AffEx (ours)          4.49      58.16  28.81  29.35     66.80   
       Masked AffEx (ours)            3.10      58.47  26.87  31.60     66.94   
       Signed AffEx (ours)            3.72      57.58  35.68  21.90     68.05   

  

In [14]:
import pandas as pd
import numpy as np

# -----------------------------
# 1) Sample dataframe (simplified version of yours)
# -----------------------------
rows = [
    ("DCAMA","Random"),
    ("DCAMA","Gaussian Noise Mask"),
    ("DCAMA","Saliency"),
    ("DCAMA","Integrated Gradients"),
    ("DCAMA","Guided IG"),
    ("DCAMA","Blur IG"),
    ("DCAMA","XRAI"),
    ("DCAMA","Deep Lift"),
    ("DCAMA","LIME"),
    ("DCAMA","Unmasked Affinity Explainer (ours)"),
    ("DCAMA","Affinity Explainer (ours)"),
    ("DMTNet","Random"),
    ("DMTNet","Gaussian Noise Mask"),
    ("DMTNet","Saliency"),
    ("DMTNet","Integrated Gradients"),
    ("DMTNet","Blur IG"),
    ("DMTNet","XRAI"),
    ("DMTNet","LIME"),
    ("DMTNet","Unmasked Affinity Explainer (ours)"),
    ("DMTNet","Affinity Explainer (ours)"),
]

cols = pd.MultiIndex.from_product(
    [
        ["COCO 20^i","Pascal 5^i"],
        ["IAUC","DAUC","Diff.","IAUC@0.01","mIoULoss@0.01","mIoULoss@0.05"]
    ],
    names=["Dataset","Metric"]
)

# Abridged numeric data (same structure as yours)
data = np.random.uniform(20, 80, size=(len(rows), len(cols)))
df = pd.DataFrame(data, index=pd.MultiIndex.from_tuples(rows, names=["Model","Explanation Method"]), columns=cols)

# -----------------------------
# 2) Define “best” direction per metric
# -----------------------------
directions = {
    ('COCO 20^i','IAUC'): None,
    ('COCO 20^i','DAUC'): None,
    ('COCO 20^i','Diff.'): 'max',
    ('COCO 20^i','IAUC@0.01'): 'max',
    ('COCO 20^i','mIoULoss@0.01'): 'min',
    ('COCO 20^i','mIoULoss@0.05'): 'min',
    ('Pascal 5^i','IAUC'): None,
    ('Pascal 5^i','DAUC'): None,
    ('Pascal 5^i','Diff.'): 'max',
    ('Pascal 5^i','IAUC@0.01'): 'max',
    ('Pascal 5^i','mIoULoss@0.01'): 'min',
    ('Pascal 5^i','mIoULoss@0.05'): 'min',
}

# -----------------------------
# 3) Compute bold winners only (no “Best” row)
# -----------------------------
def compute_best_mask(df: pd.DataFrame, directions: dict) -> pd.DataFrame:
    mask = pd.DataFrame(False, index=df.index, columns=df.columns)
    for model, group in df.groupby(level=0, sort=False):
        for col in df.columns:
            how = directions.get(col, None)
            if how is None:
                continue
            if how == 'max':
                best_val = group[col].max()
            elif how == 'min':
                best_val = group[col].min()
            else:
                continue
            mask.loc[group.index, col] = group[col] == best_val
    return mask

def latexify_bold(df: pd.DataFrame, directions: dict, floatfmt="{:.2f}"):
    """
    Return a LaTeX-ready DataFrame (as strings) where the best values are wrapped in \textbf{...}.
    """
    mask = compute_best_mask(df, directions)
    out = df.copy()
    for col in df.columns:
        for idx in df.index:
            val = df.at[idx, col]
            if pd.isna(val):
                out.at[idx, col] = ""
                continue
            s = floatfmt.format(val)
            if mask.at[idx, col]:
                s = f"\\textbf{{{s}}}"
            out.at[idx, col] = s
    return out.astype(str)

# -----------------------------
# 4) Convert to LaTeX
# -----------------------------
df_ltx = latexify_bold(final, directions, floatfmt="{:.2f}")

# Simple column alignment: first two left, rest right
colfmt = "ll" + "r"*df_ltx.shape[1]

latex_table = df_ltx.to_latex(
    escape=False,
    multirow=True,
    multicolumn=True,
    column_format=colfmt,
    bold_rows=False
)

print(latex_table)


/tmp/ipykernel_949270/1444258940.py:94: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '47.50' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  out.at[idx, col] = s
/tmp/ipykernel_949270/1444258940.py:94: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '46.85' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  out.at[idx, col] = s
/tmp/ipykernel_949270/1444258940.py:94: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.65' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  out.at[idx, col] = s
/tmp/ipykernel_949270/1444258940.py:94: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a f

\begin{tabular}{llrrrrrrrrrrrr}
\toprule
 &  & \multicolumn{6}{r}{COCO 20^i} & \multicolumn{6}{r}{Pascal 5^i} \\
 &  & IAUC & DAUC & Diff. & IAUC@0.01 & mIoULoss@0.01 & mIoULoss@0.05 & IAUC & DAUC & Diff. & IAUC@0.01 & mIoULoss@0.01 & mIoULoss@0.05 \\
Model & Explanation Method &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
\multirow[t]{12}{*}{DCAMA} & Random & 47.50 & 46.85 & 0.65 & 43.36 & 41.42 & 31.62 & 71.35 & 70.77 & 0.59 & 41.71 & 55.24 & 33.42 \\
 & Gaussian Noise Mask & 52.32 & 15.46 & 36.86 & 53.80 & 31.08 & 17.71 & 74.67 & 23.36 & 51.31 & 55.32 & 35.65 & 15.86 \\
 & Saliency & 52.02 & 41.10 & 10.92 & 44.97 & 37.01 & 23.73 & 72.99 & 66.73 & 6.27 & 44.32 & 46.42 & 24.46 \\
 & Integrated Gradients & 49.23 & 45.97 & 3.27 & 45.63 & 34.18 & 22.25 & 72.38 & 70.22 & 2.16 & 42.94 & 45.01 & 23.26 \\
 & Guided IG & 50.77 & 45.10 & 5.67 & 51.25 & 30.17 & 21.65 & 71.74 & 70.35 & 1.39 & 49.08 & 42.53 & 23.19 \\
 & Blur IG & 52.40 & 41.63 & 10.78 & 44.25 & 36.69 & 22.84 & 73.82 & 65.80 & 

In [ ]:
# turn the two level index into two columns
final_to_latex = final.reset_index()

latex_tab = tabulate(final_to_latex, headers='keys', tablefmt='latex_booktabs', showindex=False)
print(latex_tab)

\begin{tabular}{llrrrrrrrrrrrr}
\toprule
 ('Model', '')   & ('Explanation Method', '')         &   ('COCO 20\^{}i', 'IAUC') &   ('COCO 20\^{}i', 'DAUC') &   ('COCO 20\^{}i', 'Diff.') &   ('COCO 20\^{}i', 'IAUC@0.01') &   ('COCO 20\^{}i', 'mIoULoss@0.01') &   ('COCO 20\^{}i', 'mIoULoss@0.05') &   ('Pascal 5\^{}i', 'IAUC') &   ('Pascal 5\^{}i', 'DAUC') &   ('Pascal 5\^{}i', 'Diff.') &   ('Pascal 5\^{}i', 'IAUC@0.01') &   ('Pascal 5\^{}i', 'mIoULoss@0.01') &   ('Pascal 5\^{}i', 'mIoULoss@0.05') \\
\midrule
 DCAMA           & Random                             &                   47.5  &                   46.85 &                     0.65 &                        43.36 &                            41.42 &                            31.62 &                    71.35 &                    70.77 &                      0.59 &                         41.71 &                             55.24 &                             33.42 \\
 DCAMA           & Gaussian Noise Mask                &             